# 00 — Setup and the parity gate

**Run this first, and again after touching anything in `rdlib/`.**

This project's evaluation half is unusually careful: a frozen 405-row set, a
decision rule pre-registered before any number existed, paired significance
testing, and eleven committed runs. RD-22 adds a Python scoring loop beside it
so experiments can be judged in seconds instead of minutes.

That is a **second implementation**, and `eval/METHODS.md` is blunt about the
risk — two scorers can drift, and a drifted scorer produces numbers that *look*
comparable to committed runs and are not.

The repo's answer to that problem is never "be careful". It is always a check:

| existing check | what it pins |
|---|---|
| `npm run verify-viz` | the browser's reimplemented WordPiece tokenizer vs. the real `AutoTokenizer`, over 415 strings |
| RD-17's `rd17_control` | had to reproduce RD-16's `full_gloss_ft` **to the digit** before either arm could be read |
| `exact` vs `cell_lemma_ft` | pgvector vs. local brute force, 99.0% top-1 agreement |

This notebook is that move for the Python port.

In [ ]:
import sys, os
from pathlib import Path

# rdlib lives at training/rdlib; this notebook is at training/notebooks.
sys.path.insert(0, str(Path.cwd().parent))

# Cells are large and the darwin default lives under os.tmpdir(), which gets
# reaped. Point this somewhere durable and OUTSIDE the repo -- the working tree
# is in OneDrive, which would try to sync ~170 MB per cell.
os.environ.setdefault("EVAL_CELL_DIR", str(Path.home() / "rd_eval_cells"))

import rdlib
from rdlib import paths
print("repo      ", paths.REPO_ROOT)
print("cells     ", paths.cell_dir())

## Environment

Fine-tuning happens locally. A MiniLM is 22M parameters, and encoding all
117,791 WordNet synsets takes about 30 seconds on MPS — so a representation
experiment is a coffee break, not an overnight job.

In [ ]:
import platform, torch, sentence_transformers, transformers, numpy as np

print(f"python              {platform.python_version()}")
print(f"torch               {torch.__version__}")
print(f"sentence-transformers {sentence_transformers.__version__}")
print(f"transformers        {transformers.__version__}")
print(f"numpy               {np.__version__}")
print()
print(f"MPS available       {torch.backends.mps.is_available()}")
print(f"machine             {platform.machine()}")

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"\ntraining device     {DEVICE}")

## The gate

Nine checks. Each one re-derives something the TypeScript harness already
computed and asserts they agree.

The last two are the interesting ones:

- **echo** is recomputed from `query` and `results` using the port of
  `probes.ts`, then compared against the `echo` field the harness stored at run
  time. It does not trust the harness's output at all.
- **encoder parity** loads the fine-tune through `sentence-transformers` and
  compares its vectors against the ONNX model `lib/embedder.ts` actually serves,
  by shelling out to it. This is what licenses treating Python retrieval numbers
  as descriptions of production.

In [ ]:
from rdlib import parity

checks = parity.report()   # raises if anything fails

### What that proved, and what it did not

**Proved.** The scoring agrees with the harness on runs the harness produced;
the WordNet parser reproduces production's 117,791 synsets and the same gloss
`inputsSha256`; the echo rule is identical; and the Python encoder *is* the
production encoder (float32 rounding apart).

**Not proved.** That a local exact scan agrees with the live IVFFlat index.
Those are different retrieval methods and are *expected* to differ — the
approximate index costs roughly 0.3 points of lenient R@1. A cell scan is the
right instrument for "is this representation better" and the wrong one for
"what will users get".

**The rule this establishes:** Python numbers are for iteration. Anything that
would change a decision gets confirmed through `npx tsx scripts/eval.ts`.
Notebook `04` is where that reconciliation happens.

## The strongest check: a full round-trip

Build a cell in Python, score it **both** ways, compare per query. When RD-22
was built this returned 287/287 identical on deep rank, top-1, and full top-10
order — not "close", identical.

It is not part of `report()` because it needs a built cell and a harness run,
which the gate deliberately does not require. Run it once after changing
`retrieval.py`.

In [ ]:
from rdlib.build import build_wordnet_cell

# ~30s on MPS. This is RD-16's `full_gloss_ft` control, rebuilt from scratch.
info = build_wordnet_cell("rd22_gloss_ft")
info

Now score the same cell with the TypeScript harness, from the repo root:

```bash
EVAL_CELL_DIR=~/rd_eval_cells npx tsx scripts/eval.ts \
    --set eval/sets/v1.jsonl --index-file rd22_gloss_ft --tag rd22_roundtrip
```

It should print **lenient R@1 25.4% / strict 21.6% / R@10 51.6% / MRR 0.304 /
echo 14.6%** — RD-16's control, to the digit. Then:

In [ ]:
print(parity.check_cell_roundtrip("rd22_gloss_ft", "rd22_roundtrip"))

## Where things live

| path | what |
|---|---|
| `eval/sets/v1.jsonl` | the frozen benchmark. 405 rows, sha256-gated on load |
| `eval/runs/*.json` | committed evidence. `prod_wikt_shipped` is current production |
| `eval/runs/*.shortlist.jsonl` | RD-12 sidecars: 405 × 100 candidates **with gloss text** |
| `eval/METHODS.md` | the lab notebook. §9a is the decision rule that binds you |
| `~/rd_eval_cells` | vector cells. Gitignored, ~170 MB each, never committed |
| `training/artifacts/` | checkpoints and generated pairs. Gitignored |

**Next:** `01_explore_the_evidence.ipynb` — look at the failures before touching
a model.